# 01 — Basic Damage Simulation

Run a single scenario (Warrior vs NPC) and inspect the damage distribution.

In [ ]:
import logging
from pathlib import Path

from omega.model.constants import SKILLID_SWORDSMANSHIP, SKILLID_TACTICS, SKILLID_ANATOMY, SKILLID_WRESTLING
from omega.shard import ShardData
from omega.simulation import (
    ArmorSpec, CombatantSpec, Scenario, WeaponSpec, run_scenario,
)
from omega.reporting.tables import summary_table, format_table_html
from omega.reporting.plots import damage_histogram, damage_breakdown
from omega.logging import setup_logging

# Suppress noisy stub warnings — only show errors in notebook output
setup_logging(level=logging.ERROR)

# Works whether CWD is the project root or the notebooks/ directory
SHARD_ROOT = Path("submodules/zuluhotel_omega_2.5")
if not SHARD_ROOT.exists():
    SHARD_ROOT = Path("../submodules/zuluhotel_omega_2.5")
shard = ShardData.from_path(SHARD_ROOT)

In [ ]:
scenario = Scenario(
    attacker=CombatantSpec(
        name="Warrior",
        skills={SKILLID_SWORDSMANSHIP: 100, SKILLID_TACTICS: 100, SKILLID_ANATOMY: 100},
        str_=100, dex_=100, int_=25,
        class_levels={"IsWarrior": 5},
        weapon=WeaponSpec(name="Broadsword", damage="3d6+2", attribute=SKILLID_SWORDSMANSHIP),
    ),
    defender=CombatantSpec(
        name="Target Dummy",
        is_npc=True,
        str_=50, dex_=50, int_=50,
        hp=500,
        skills={SKILLID_WRESTLING: 60},  # NPC combat skill — affects hit chance
        armor=ArmorSpec(name="Plate", ar=30),
    ),
    iterations=200,
    base_seed=42,
)

result = run_scenario(scenario, shard=shard)
print(f"Successes: {result.success_count}/{result.iteration_count}")
print(f"Hit rate: {result.ratios.hit_rate:.1%}")
print(f"Mean damage (overall): {result.damage_stats.mean:.2f}")
print(f"Mean damage (on hit):  {result.damage_stats_on_hit.mean:.2f}")
print(f"Range: {result.damage_stats.min:.0f} – {result.damage_stats.max:.0f}")

In [ ]:
# Damage distribution histogram
damage_histogram(result, title="Warrior vs NPC — 200 hits")

In [ ]:
# Base / absorbed / final breakdown
damage_breakdown(result, title="Damage Breakdown")

In [ ]:
# HTML summary table — overall stats (including misses) and on-hit stats
from IPython.display import HTML
from omega.simulation.stats import SimulationResult

rows = summary_table(
    SimulationResult(cells=[result]),
    stats=["mean", "mean_on_hit", "median", "min", "max", "p5", "p95", "hit_rate", "count"],
)
HTML(format_table_html(rows))

## Enchanted Weapon Comparison (V1.5)

Apply an enchantment using `enchant_with()` and compare against the plain weapon.

In [ ]:
from omega.config.enchantments import Enchantment
from omega.reporting.plots import comparison_overlay, enchantment_comparison
from omega.reporting.tables import comparison_table

# Create an enchanted version of the same weapon.
# ChanceOfEffect controls how often the spell fires (75 = 75%);
# EffectCircle controls spell power (circle 10 = highest tier).
plain_sword = WeaponSpec(name="Broadsword", damage="3d6+2", attribute=SKILLID_SWORDSMANSHIP)
fire_sword = WeaponSpec(
    name="Broadsword", damage="3d6+2", attribute=SKILLID_SWORDSMANSHIP,
    properties={"ChanceOfEffect": 75, "EffectCircle": 10},
).enchant_with(Enchantment.OF_DAEMONS_BREATH)  # Fireball on hit

print(f"Plain:     hitscript={plain_sword.hitscript}")
print(f"Enchanted: hitscript={fire_sword.hitscript}")
print(f"           properties={fire_sword.properties}")

In [ ]:
RUN_KW = dict(shard=shard)

attacker = CombatantSpec(
    name="Warrior",
    skills={SKILLID_SWORDSMANSHIP: 100, SKILLID_TACTICS: 100, SKILLID_ANATOMY: 100},
    str_=100, dex_=100, int_=25,
    class_levels={"IsWarrior": 5},
)
defender = CombatantSpec(
    name="Target Dummy", is_npc=True,
    str_=50, dex_=50, int_=50, hp=500,
    skills={SKILLID_WRESTLING: 60},
    armor=ArmorSpec(name="Plate", ar=30),
)

import dataclasses

enchant_results = {}
for label, weapon in [("Plain", plain_sword), ("Daemon's Breath", fire_sword)]:
    atk = dataclasses.replace(attacker, weapon=weapon)
    enchant_results[label] = run_scenario(
        Scenario(attacker=atk, defender=defender, iterations=200, base_seed=42),
        **RUN_KW,
    )
    ds = enchant_results[label].damage_stats
    r = enchant_results[label].ratios
    print(f"  {label:20s}  mean={ds.mean:6.2f}  hit_rate={r.hit_rate:.1%}  "
          f"spell_strike={r.spell_strike_rate:.1%} (on hit: {r.spell_strike_rate_on_hit:.1%})")

In [ ]:
# Overlaid histograms: plain vs enchanted
comparison_overlay(enchant_results, title="Plain vs Daemon's Breath Enchantment")

In [ ]:
# Side-by-side stat comparison with enchantment-specific columns
rows = comparison_table(
    enchant_results,
    stats=["mean", "mean_on_hit", "median", "min", "max", "p5", "p95",
           "hit_rate", "spell_strike_rate", "spell_strike_rate_on_hit"],
)
HTML(format_table_html(rows))